# Is MoM-BH\*-1 near its Eddington limit? Checks on your stage 4 result (stage 5)

**The question:** stage 4 gave MoM-BH\*-1 a total mass of about 1.7 million Suns. That number was not measured. It comes from *assuming* that the object shines at about its Eddington limit. The discovery team now argues that typical little red dots shine 5–50 times *above* their limit (Naidu et al. 2026b; Sun et al. 2026). If MoM-BH\*-1 does too, it is 5–50 times lighter. So two things need checking:
1. Is the near-limit assumption allowed for this object?
2. Does your black-hole mass depend on it?

**Two published ways to weigh an object without that assumption** (both from Naidu et al. 2026b, arXiv:2606.30711):
- **Surface gravity:** M = g R² / G, with the gravity g read from the shape of the spectrum.
- **Gas speed:** gas that escapes from an object moves at least at the escape speed, √(2GM/R). So the speed v of outflowing gas gives M ≤ v² R / (2G).

**Important:** Naidu et al. treat both as **upper limits** on the mass, because they picture the glowing surface as an outflowing wind. Keep that in mind in every step.

**The steps:**
- Step 1: your numbers.
- Step 2: surface gravity, and where MoM-BH\*-1 sits among 87 little red dots.
- Step 3: gas speed.
- Step 4: both checks side by side.
- Step 5: how robust is your black-hole mass?
- Step 6: the figure for your note.

**How to use:** run the grey code cells from top to bottom with **Shift + Enter** (or *Runtime → Run all*). It takes about a minute.

## Step 1 — Set up: constants and your numbers

This cell stores the constants, your stage 4 results, your measurements, and the published gas speeds.

**Check the first block against your own stage 4 output** (Step 6, the block called "COMBINED (headline)"). The numbers below are the ones you sent me. If yours differ, change them here.

**The published speeds:** Naidu et al. (2026a, the discovery paper) fitted the Hβ line with one emission line and three absorption components: one at the line centre and one on each side, at **−1,532 (+345/−113) km/s** and **+1,556 (+232/−1,378) km/s**. Minus means gas moving towards us (away from the object: an outflow). Plus means moving away from us (towards the object: infall). Read the error bars like this: −1,532 (+345/−113) means "between −1,645 and −1,187".

**Before you cite these speeds, look them up yourself** in the paper's Extended Data table of line fits (the paper is free to read).

**Error bars in this notebook:** the function `spread()` draws 400,000 possible values of a measurement inside its error bar. Any quantity calculated from those draws then gets its own range automatically (the middle value and the range holding the middle 68%). The radius comes from your luminosity and temperature, R = √(L / 4πσT⁴), so everything uses one consistent set of numbers.

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar, brentq

# --- constants (cgs units) ---
G, c, sigma_sb = 6.674e-8, 2.998e10, 5.6704e-5
Msun, Lsun, AU, KM = 1.989e33, 3.828e33, 1.496e13, 1e5
KAPPA_ES = 0.34                       # how strongly light pushes on ionised gas, cm^2/g (as in stages 2-4)

def eddington_mass(L):
    # the mass (in Suns) whose Eddington limit equals the luminosity L (erg/s)
    return KAPPA_ES * L / (4*np.pi*G*c) / Msun

def radius(L, T):
    # radius of the glowing surface (cm) from the Stefan-Boltzmann law
    return np.sqrt(L / (4*np.pi*sigma_sb*T**4))

# --- your stage 4 results (Step 6, "COMBINED" block): low, middle, high of the 68% range, in Suns ---
M_TOTAL = (1.25e6, 1.70e6, 2.35e6)    # total mass
M_BH = (6354, 11446, 22895)           # black hole

# --- your measurements ---
L_OBS, SIG_LOGL = 2.9e44, 0.1         # luminosity (erg/s), +/- 0.1 in log
T_OBS, SIG_T = 5200, 300              # surface temperature, K

# --- published by Naidu et al. (2026a): speeds of the two side components of H-beta, km/s: low, middle, high ---
V_BLUE = (1187, 1532, 1645)           # gas moving towards us:   -1,532 (+345/-113)
V_RED = (178, 1556, 1788)             # gas moving away from us: +1,556 (+232/-1,378)

# --- random draws inside the error bars ---
N_DRAW = 400_000
rng = np.random.default_rng(7)        # fixed seed: you get the same numbers every run

def spread(low, mid, high):
    # draws of a quantity whose 68% range is low-high around mid (the two sides can differ)
    z = rng.standard_normal(N_DRAW)
    return np.where(z < 0, mid + z*(mid - low), mid + z*(high - mid))

def middle_and_range(x):
    lo, mid, hi = np.percentile(x, [16, 50, 84])
    return mid, lo, hi

T_draw = spread(T_OBS - SIG_T, T_OBS, T_OBS + SIG_T)
L_draw = L_OBS * 10**spread(-SIG_LOGL, 0.0, SIG_LOGL)
R_draw = radius(L_draw, T_draw)                      # cm
v_draw = np.abs(spread(*V_BLUE))                     # km/s

mid, lo, hi = middle_and_range(R_draw/AU)
print(f"Your total mass:   {M_TOTAL[1]:.2e} Suns (range {M_TOTAL[0]:.2e} - {M_TOTAL[2]:.2e})")
print(f"Your black hole:   {M_BH[1]:,} Suns (range {M_BH[0]:,} - {M_BH[2]:,})")
print(f"Your radius:       {mid:,.0f} AU (range {lo:,.0f} - {hi:,.0f})")
print(f"Eddington mass for your luminosity: {eddington_mass(L_OBS):.2e} Suns")

# --- plot style and colours (same as stages 2-4) ---
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b", "axes.titlecolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 11, "legend.frameon": False, "legend.fontsize": 9.5})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, ZONE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#ebeae5", "#fcfcfb"

## Step 2 — Surface gravity: where does MoM-BH\*-1 sit among 87 little red dots?

Curtis et al. (2026) fitted star-like atmosphere models to 87 little red dots and published the results (you loaded this table in stages 2 and 3). The cell works out, for each object:
- its **radius**, from their luminosity and temperature;
- its **mass from surface gravity**, M = g R² / G (as in stage 2, Step 12);
- its **Eddington ratio** φ: how many times above its Eddington limit it shines. This is their own column.

It also saves a copy of the table (their table may change later, so keep the copy with your notebooks).

**Expect:** the typical object shines about 11 times above its limit. MoM-BH\*-1 comes out at about 2.3 times, with a long error bar (0.9–13).

**The plot:** luminosity across, gravity mass up. The three grey lines mark objects shining exactly at their limit, 10 times above it, and 100 times above it.

In [ ]:
URL = "https://raw.githubusercontent.com/o-curtis/lrd-quasistar-fits/main/summary/photospheric_parameters.csv"
table = pd.read_csv(URL)
table.to_csv("curtis_table_copy.csv", index=False)
print(f"Table downloaded on {datetime.date.today()} and saved as curtis_table_copy.csv\n")

main = table[table["family"].isin(["primary", "supplementary/prism", "supplementary/desi",
                                   "supplementary/grating"])].drop_duplicates("target").copy()
main["L"] = 10**main.logL_50 * Lsun                                    # luminosity, erg/s
main["R_au"] = radius(main.L, main.teff_50) / AU                       # radius, AU
main["mass"] = 10**main.logg_50 * (main.R_au*AU)**2 / G / Msun         # mass from gravity, Suns
is_mom = (main["target"] == "MoMBH1").values
mom = main[is_mom].iloc[0]
others = main[~is_mom]

# MoM-BH*-1's gravity mass: low, middle, high (from the low/middle/high gravity, with their radius)
M_GRAV = tuple(10**mom[f"logg_{p}"] * (mom.R_au*AU)**2 / G / Msun for p in ("16", "50", "84"))

print(f"{len(main)} little red dots loaded")
print(f"Typical little red dot: shines {main.phi_50.median():.0f} times above its limit (gravity mass {main.mass.median():.1e} Suns)")
print(f"MoM-BH*-1:              shines {mom.phi_50:.1f} times above its limit, range {mom.phi_16:.1f} - {mom.phi_84:.1f} "
      f"(gravity mass {M_GRAV[1]:.1e} Suns, range {M_GRAV[0]:.1e} - {M_GRAV[2]:.1e})")
print(f"Their fit for MoM-BH*-1: {mom.teff_50:,.0f} K, {mom.L:.1e} erg/s, radius {mom.R_au:,.0f} AU, log g = {mom.logg_50:.2f}\n")
print(f"Objects whose middle value is closer to the limit than MoM-BH*-1's: {np.mean(main.phi_50 < mom.phi_50)*100:.0f}%")
print(f"Objects that could be within the classic range (below 1.58) inside their error bar: "
      f"{int((main.phi_16 < 1.58).sum())} of {len(main)}, MoM-BH*-1 included")
odd = main[(main.logL_50 > 13) | (main.logL_50 < 8)]
print(f"Fits with an implausible luminosity (treat with care): {', '.join(odd.target)}")
ok = (len(main) == 87) and (2.2e6 < M_GRAV[1] < 2.6e6)
print(f"\n{'PASS' if ok else 'FAIL'}  87 objects loaded and MoM-BH*-1's gravity mass matches stage 2, Step 12 (2.4 million Suns)")

fig, ax = plt.subplots(figsize=(8.5, 5.6))
ax.set_axisbelow(True)
L_line = np.logspace(41, 47, 50)
for ratio, word in [(1, "at its limit"), (10, "10× above"), (100, "100× above")]:
    ax.plot(L_line, eddington_mass(L_line)/ratio, color=MUTED, lw=1)
    ax.text(2.6e45, eddington_mass(2.6e45)/ratio*1.35, word, color=INK2, fontsize=9.5, ha="right",
            rotation=33, rotation_mode="anchor")
ax.plot(others.L, others.mass, "o", color=BLUE, ms=6.5, alpha=0.75, mec=SURFACE, mew=0.8,
        label="little red dots (Curtis et al. 2026)")
ax.errorbar(mom.L, M_GRAV[1], yerr=[[M_GRAV[1]-M_GRAV[0]], [M_GRAV[2]-M_GRAV[1]]], fmt="s", color=ORANGE,
            ms=9, mec=SURFACE, mew=1.2, lw=2, label="MoM-BH*-1 (Curtis et al. 2026)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1e42, 3e45); ax.set_ylim(1e2, 1e8)
outside = int(((main.L < 1e42) | (main.L > 3e45) | (main.mass < 1e2) | (main.mass > 1e8)).sum())
ax.set_xlabel("Luminosity (erg/s)")
ax.set_ylabel("Mass from surface gravity (Suns)")
ax.set_title("Surface gravity puts most little red dots far above their limit", loc="left")
ax.legend(loc="upper left")
plt.tight_layout(); fig.savefig("s5_step2_gravity.png", dpi=150); plt.show()
print(f"({outside} of the {len(main)} objects fall outside the plotted area.)")

**What it means:** by surface gravity, MoM-BH\*-1 *could* be near its limit: its ratio is 2.3, and its error bar reaches down to 0.9. The typical object sits at about 11.

**Be careful how you say this:**
- **It is not special by this test alone.** The gravity error bars are long, so many other objects could also be near their limit (the cell prints how many).
- **Its mass rank is mostly its brightness.** It is one of the most luminous objects in the table, so a normal ratio already gives a large mass.
- **In the discovery team's picture this is an upper limit on the mass**, because a wind makes the gravity look stronger than it is (Naidu et al. 2026b).
- **Their fit is not your fit.** They find a hotter, brighter surface (6,465 K) because they allow for dust. The ratio 2.3 belongs to their numbers.

## Step 3 — Gas speed

**The physics:** to leave an object of mass M from radius R, gas needs the escape speed, **v = √(2GM/R)**. So gas that is escaping at speed v tells you **M ≤ v² R / (2G)**: an upper limit. (Infalling gas works the other way and gives a lower limit, but the infalling component here is too poorly measured to help.)

**What the cell does:**
- It works out the escape speed your near-limit envelope would have. **Expect about 1,380 km/s.**
- It turns the measured outflow speed (1,532 km/s) into an upper limit on the mass. **Expect about 2.0 million Suns.**
- It repeats this the way the discovery team does for typical objects, where a wind moves at 3–5 times the escape speed. Then the mass is 9–25 times smaller.

Nothing in stage 4 used these speeds.

In [ ]:
# the envelope stage 4 assumed: total mass = Eddington mass / f, with f anywhere between 0.76 and 1.58
f_draw = rng.uniform(0.76, 1.58, N_DRAW)
M_near = eddington_mass(L_draw) / f_draw
V_PRED = middle_and_range(np.sqrt(2*G*M_near*Msun / R_draw) / KM)

# upper limit on the mass from the outflowing gas, and the Eddington ratio that goes with it
M_SPEED = middle_and_range((v_draw*KM)**2 * R_draw / (2*G) / Msun)
RATIO_SPEED = middle_and_range(eddington_mass(L_draw) / ((v_draw*KM)**2 * R_draw / (2*G) / Msun))

# the escape speed if it shone 5-50 times above its limit
V_LIGHT = tuple(np.sqrt(2*G*eddington_mass(L_OBS)/k*Msun / radius(L_OBS, T_OBS)) / KM for k in (50, 5))

print(f"Escape speed of your near-limit envelope:  {V_PRED[0]:,.0f} km/s  (range {V_PRED[1]:,.0f} - {V_PRED[2]:,.0f})")
print(f"Measured, gas moving towards us:           {V_BLUE[1]:,} km/s  (range {V_BLUE[0]:,} - {V_BLUE[2]:,})")
print(f"Measured, gas moving away from us:         {V_RED[1]:,} km/s  (range {V_RED[0]:,} - {V_RED[2]:,})")
print(f"Escape speed if 5-50 times above its limit: {V_LIGHT[0]:,.0f} - {V_LIGHT[1]:,.0f} km/s\n")
print(f"Upper limit on the mass from the outflow:  {M_SPEED[0]:.2e} Suns  (range {M_SPEED[1]:.2e} - {M_SPEED[2]:.2e})")
print(f"So it shines at least {RATIO_SPEED[0]:.1f} times its Eddington limit (range {RATIO_SPEED[1]:.1f} - {RATIO_SPEED[2]:.1f})")
print(f"If the outflow is a wind at 3-5 times the escape speed: mass {M_SPEED[0]/25:.1e} - {M_SPEED[0]/9:.1e} Suns, "
      f"{RATIO_SPEED[0]*9:.0f} - {RATIO_SPEED[0]*25:.0f} times above its limit")

rows = ["Escape speed of your near-limit envelope", "Measured: gas moving towards us",
        "Measured: gas moving away from us", "Escape speed if 5–50× above its limit"]
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.set_axisbelow(True)
ax.barh(3, V_PRED[2]-V_PRED[1], left=V_PRED[1], height=0.24, color=BLUE)
ax.plot(V_PRED[0], 3, "o", color=SURFACE, ms=6)
ax.errorbar(V_BLUE[1], 2, xerr=[[V_BLUE[1]-V_BLUE[0]], [V_BLUE[2]-V_BLUE[1]]], fmt="o", color=ORANGE, ms=9,
            mec=SURFACE, mew=1.5, lw=2, capsize=4)
ax.errorbar(V_RED[1], 1, xerr=[[V_RED[1]-V_RED[0]], [V_RED[2]-V_RED[1]]], fmt="o", color=ORANGE, ms=9,
            mec=SURFACE, mew=1.5, lw=2, capsize=4)
ax.barh(0, V_LIGHT[1]-V_LIGHT[0], left=V_LIGHT[0], height=0.24, color=MUTED)
ax.set_yticks([3, 2, 1, 0]); ax.set_yticklabels(rows)
ax.tick_params(axis="y", length=0, labelcolor=INK)
ax.grid(axis="y", visible=False)
ax.set_xlim(0, 2000); ax.set_ylim(-0.6, 3.6)
ax.set_xlabel("Speed (km/s)")
ax.set_title("The fastest gas moves at about the escape speed of a near-limit object", loc="left")
plt.tight_layout(); fig.savefig("s5_step3_gas_speed.png", dpi=150); plt.show()

**What it means:** the outflowing gas (1,532 km/s) moves at about the escape speed your near-limit envelope would have (about 1,380 km/s). So this test does not force MoM-BH\*-1 to be lighter than you assumed. That is all it shows.

**Four cautions. Put them in anything you write:**
1. **It is an upper limit, not a measurement.** An outflow only needs to be *at least* as fast as the escape speed. The discovery team argues that winds in these objects move at 3–5 times the escape speed (Naidu et al. 2026b). Then MoM-BH\*-1 would weigh about 100,000–200,000 Suns and shine 9–25 times above its limit, like a typical little red dot.
2. **The discovery paper does not read these features as moving gas.** Naidu et al. (2026a) say the Hβ profile is symmetric, which makes inflows and outflows unlikely. They prefer one shell of gas, possibly with light scattering back and forth inside it.
3. **The infalling component can't help.** Infalling gas would give a *lower* limit on the mass, but its speed is only known to lie between 178 and 1,788 km/s.
4. **With Curtis et al.'s hotter, brighter surface the match goes away.** Step 4 shows this.

A way to tell moving gas from scattering: moving gas shows the same speed in every spectral line; scattering generally doesn't. That is the Hγ test for the full paper.

## Step 4 — Both checks side by side

To compare fairly, everything is turned into one quantity: **how many times above its Eddington limit the object shines** (the Eddington ratio). The cell does this for two versions of the glowing surface:
- **yours:** 5,200 K and 2.9 × 10⁴⁴ erg/s (blackbody fit, no dust);
- **Curtis et al.'s:** 6,465 K and 8.4 × 10⁴⁴ erg/s (atmosphere models, with dust).

For the gas speed, the ratio is a *lower* limit (because the mass is an upper limit). Your classic model needs a ratio between 0.76 and 1.58.

In [ ]:
# gas speed with Curtis et al.'s surface
T_c = spread(mom.teff_16, mom.teff_50, mom.teff_84)
L_c = 10**spread(mom.logL_16, mom.logL_50, mom.logL_84) * Lsun
R_c = radius(L_c, T_c)
RATIO_SPEED_C = middle_and_range(eddington_mass(L_c) / ((v_draw*KM)**2 * R_c / (2*G) / Msun))
RATIO_GRAV_C = (mom.phi_50, mom.phi_16, mom.phi_84)                   # their own column

print("How many times above its Eddington limit?        your surface          Curtis et al.'s surface")
print(f"  from surface gravity (their fit)                 -                   {RATIO_GRAV_C[0]:.1f}  ({RATIO_GRAV_C[1]:.1f} - {RATIO_GRAV_C[2]:.1f})")
print(f"  from the outflow speed (at least)             {RATIO_SPEED[0]:.1f}  ({RATIO_SPEED[1]:.1f} - {RATIO_SPEED[2]:.1f})        {RATIO_SPEED_C[0]:.1f}  ({RATIO_SPEED_C[1]:.1f} - {RATIO_SPEED_C[2]:.1f})")
print(f"  ... if it is a wind at 3-5x the escape speed  {RATIO_SPEED[0]*9:.0f} - {RATIO_SPEED[0]*25:.0f}                {RATIO_SPEED_C[0]*9:.0f} - {RATIO_SPEED_C[0]*25:.0f}")
print(f"  your classic model needs                      0.76 - 1.58")
print(f"  typical little red dot                        about {main.phi_50.median():.0f} (Curtis et al.'s table); 5 - 50 (Sun et al. 2026)\n")

allowed = (RATIO_SPEED[1] < 1.58) and (RATIO_GRAV_C[1] < 1.58)
print(f"{'ALLOWED' if allowed else 'NOT ALLOWED'}: with your surface, the outflow speed is compatible with the classic range, "
      f"and the gravity reaches it inside its error bar.")
print("NOT SHOWN: both are upper limits on the mass if the surface is an outflow, and with Curtis et al.'s surface\n"
      "           both point to about 2.4, above the classic range.")

**What it means, in one sentence:** a near-limit envelope is **allowed** for MoM-BH\*-1, but it is **not shown**.

- With your surface, the outflow speed gives a ratio of at least about 1, which fits the classic range.
- With Curtis et al.'s surface, gravity and outflow speed both give about 2.4. That is above the classic range (0.76–1.58), though still far below the typical 11.
- If the outflow is a fast wind, the ratio is 9–25 or more, and MoM-BH\*-1 is an ordinary little red dot.

**What none of this tells you:** how the mass is split between black hole and gas. All these tests weigh everything inside the glowing surface.

## Step 5 — How robust is your black-hole mass?

Four tests, using the simulator itself. (Stage 4, Step 7 showed the network and the simulator give the same answer, so the network isn't needed here.)

1. **Reproduce stage 4.** **PASS** if the middle value and both ends of the range agree with your stage 4 numbers within 5%.
2. **Curtis et al.'s hotter, brighter surface** instead of yours.
3. **Drop the near-limit assumption.** Ask the simulator what black hole gives 5,200 K if the object shines 1, 2, 3 or 5 times above its limit (so it is 1, 2, 3 or 5 times lighter than the Eddington mass).
4. **The temperature floor.** In Ball et al.'s full model, the surface stops cooling at about 4,500 K once the black hole holds more than about 5% of the mass. Your simple model has no such floor. How far is your 5,200 K above it?

In [ ]:
# ---- the simulator, copied from stage 4, Step 2 ----
GAMMA, n = 4/3, 3
MODEL_ERR = 0.10
PRIOR = {"log_total": (5.5, 8.0), "log_bh": (1.5, 7.5), "log_eta": (np.log10(0.03), np.log10(0.3)), "f": (0.76, 1.58)}

def lane_emden(xi, y):
    theta, phi = y
    return [-phi / xi**2, xi**2 * max(theta, 0.0)**n]

def at_surface(xi, y):
    return y[0]
at_surface.terminal, at_surface.direction = True, -1

def loaded_polytrope(xi0):
    phi_bh = xi0 / (2*n)
    sol = solve_ivp(lane_emden, [xi0, 100], [1.0, phi_bh + (2/3)*xi0**3], events=at_surface, rtol=1e-9, atol=1e-12)
    return phi_bh, sol.y_events[0][0][1], sol.t_events[0][0]

best = minimize_scalar(lambda x: -loaded_polytrope(x)[0]/loaded_polytrope(x)[1], bounds=(0.8, 1.2),
                       method="bounded", options={"xatol": 1e-7})
XI0_MAX, Q_MAX = best.x, -best.fun
_xi0 = np.geomspace(1e-7, XI0_MAX, 300)
_tab = np.array([loaded_polytrope(x) for x in _xi0])
_q, _phi_s, _xi1 = _tab[:, 0]/_tab[:, 1], _tab[:, 1], _tab[:, 2]

def simulate(log_total, log_bh, log_eta, f):
    M_tot, M_bh, eta = 10**log_total*Msun, 10**log_bh*Msun, 10**log_eta
    q = M_bh / M_tot
    valid = q < Q_MAX
    lq = np.log(np.clip(q, 1e-12, 0.999999*Q_MAX))
    phi_s = np.exp(np.interp(lq, np.log(_q), np.log(_phi_s)))
    xi1 = np.interp(lq, np.log(_q), _xi1)
    K = np.pi*G * (M_tot / (4*np.pi*phi_s))**(2/3)
    L = f * 4*np.pi*G*c*M_tot / KAPPA_ES
    A = (16*np.pi*eta/GAMMA) * (G*M_bh)**2 / np.sqrt(GAMMA*K)
    R = xi1 * np.sqrt(K/(np.pi*G)) * (L/A)**(-2/5)
    T = (L / (4*np.pi*sigma_sb*R**2))**0.25
    return np.log10(L), np.where(valid, np.log10(T), np.nan), valid

# ---- the inference, as in stage 4, Step 6, but with the measurements as inputs ----
CAL_SHARE = np.array([5e-4, 0.01, 0.02, 0.05, 0.1])
CAL_RATIO = np.array([1.10, 0.83, 0.73, 0.61, 0.68])

def weighted_range(x, w):
    order = np.argsort(x)
    cum = np.cumsum(w[order]); cum /= cum[-1]
    return np.interp([0.16, 0.5, 0.84], cum, x[order])

def infer(L, T, sig_T, sig_logL=0.1, N=2_000_000, seed=5):
    draw_rng = np.random.default_rng(seed)
    sig_logT = np.sqrt((sig_T / T / np.log(10))**2 + np.log10(1 + MODEL_ERR)**2)
    out = {}
    for label, calibrate in [("as is", False), ("calibrated", True)]:
        P = np.column_stack([draw_rng.uniform(*PRIOR[k], N) for k in PRIOR])
        logL, logT, valid = simulate(*P.T)
        if calibrate:
            logT = logT - np.log10(np.interp(P[:, 1] - P[:, 0], np.log10(CAL_SHARE), CAL_RATIO))
        chi2 = ((logL - np.log10(L))/sig_logL)**2 + ((logT - np.log10(T))/sig_logT)**2
        w = np.where(valid, np.exp(-0.5*np.nan_to_num(chi2, nan=1e9)), 0.0)
        out[label] = (P, w / w.sum())
    out["combined"] = (np.vstack([out["as is"][0], out["calibrated"][0]]),
                       np.concatenate([out["as is"][1], out["calibrated"][1]]))
    return out

def show(result, label):
    P, w = result["combined"]
    bh = 10**weighted_range(P[:, 1], w)
    tot = 10**weighted_range(P[:, 0], w)
    print(f"--- {label} ---")
    print(f"   black hole: {bh[1]:,.0f} Suns   (68% range {bh[0]:,.0f} - {bh[2]:,.0f})")
    print(f"   total mass: {tot[1]:.2e} Suns (68% range {tot[0]:.2e} - {tot[2]:.2e})")
    return bh, tot

# ---- test 1: reproduce stage 4 ----
print(f"Black-hole limit: {Q_MAX:.4f} (stage 4 gave 0.1015)\n")
res_mine = infer(L_OBS, T_OBS, SIG_T)
bh_mine, tot_mine = show(res_mine, f"test 1, your measurements: {T_OBS:,} K, {L_OBS:.1e} erg/s")
worst = max(abs(bh_mine[i]/M_BH[i] - 1) for i in range(3))
print(f"{'PASS' if worst < 0.05 else 'FAIL'}  matches your stage 4 black-hole mass within {100*worst:.1f}%\n")

# ---- test 2: Curtis et al.'s surface ----
res_curtis = infer(mom.L, mom.teff_50, 230)
bh_curtis, tot_curtis = show(res_curtis, f"test 2, Curtis et al.'s fit: {mom.teff_50:,.0f} K, {mom.L:.1e} erg/s")

# ---- test 3: drop the near-limit assumption ----
def black_hole_for(T, L, eta, f):
    # the black hole (Suns) that gives surface temperature T, for total mass = Eddington mass / f (model as is)
    log_total = np.log10(eddington_mass(L) / f)
    gap = lambda log_bh: 10**simulate(np.array([log_total]), np.array([log_bh]), np.array([np.log10(eta)]), np.array([f]))[1][0] - T
    return 10**brentq(gap, 1.0, log_total + np.log10(0.999*Q_MAX))

print("\n--- test 3, the black hole that gives 5,200 K (model as is, eta = 0.1) ---")
for f in [1, 2, 3, 5]:
    bh = black_hole_for(T_OBS, L_OBS, 0.1, f)
    print(f"   {f} times above its limit: total {eddington_mass(L_OBS)/f:.1e} Suns, black hole {bh:,.0f} Suns "
          f"(share {bh/(eddington_mass(L_OBS)/f):.3f})")
b0 = black_hole_for(T_OBS, L_OBS, 0.1, 1.2)
print(f"   For comparison, eta = 0.03 gives {black_hole_for(T_OBS, L_OBS, 0.03, 1.2):,.0f} Suns and eta = 0.3 gives "
      f"{black_hole_for(T_OBS, L_OBS, 0.3, 1.2):,.0f} Suns.")
step = 1.02
print(f"   Scaling: black hole = {b0:,.0f} Suns x (T/5,200 K)^{np.log(black_hole_for(T_OBS*step, L_OBS, 0.1, 1.2)/b0)/np.log(step):.2f} "
      f"x (L/2.9e44)^{np.log(black_hole_for(T_OBS, L_OBS*step, 0.1, 1.2)/b0)/np.log(step):.2f} "
      f"x (eta/0.1)^{np.log(black_hole_for(T_OBS, L_OBS, 0.1*step, 1.2)/b0)/np.log(step):.2f} "
      f"x (f/1.2)^{np.log(black_hole_for(T_OBS, L_OBS, 0.1, 1.2*step)/b0)/np.log(step):.2f}")

# ---- test 4: the temperature floor in Ball et al.'s full model ----
T_FLOOR = 4500
print(f"\n--- test 4, the temperature floor ---")
print(f"   Ball et al.'s model levels off near {T_FLOOR:,} K. Your {T_OBS:,} +/- {SIG_T} K is {(T_OBS - T_FLOOR)/SIG_T:.1f} error bars above it.")
print(f"   Largest black hole a classic quasi-star of your total mass can hold: {Q_MAX*M_TOTAL[1]:.1e} Suns")

**What it means:**

- **Test 2:** the hotter, brighter fit moves the black hole from about 11,000 to about 15,000 Suns, inside your error bar. (The total mass rises to about 5 million Suns, because the total follows the luminosity.)
- **Test 3, the useful surprise:** the black-hole mass hardly depends on the near-limit assumption. Even at 5 times above the limit it is about 12,000 Suns. The scaling line shows why: the black hole goes only as the fourth root of f. So your result of roughly 10,000 Suns **does not need** Steps 2–4 to come out a particular way.
- **Test 3 also shows what the error bar is made of:** mostly the temperature and the assumed range of the efficiency η.
- **Test 4, the weak point:** your 5,200 K is only about two error bars above the temperature at which Ball et al.'s model stops cooling. If a heavier quasi-star has the same floor, black holes up to the classic limit (about 200,000 Suns) are not ruled out. So write **"about 10,000 Suns; up to about 200,000 not excluded"**.

And all of this holds only inside the classic model. Newer models (Coughlin & Begelman 2024; Santarelli et al. 2026) allow a much bigger black hole and stay near 6,000 K whatever its mass.

## Step 6 — The figure for your note

One figure, two panels, saved as **figure1_MoM-BH1.png** (in the Colab file list on the left: right-click → Download).

- **Left:** the Eddington ratio from the two checks (Step 4). Filled symbols use your surface, open symbols use Curtis et al.'s. The green band is the range your classic model needs; the grey band is the range for typical objects (Sun et al. 2026). The blue dots are the other 86 little red dots from Curtis et al.'s table.
- **Right:** your black-hole mass from Step 5, test 1. The dashed line is the largest black hole a classic quasi-star of your total mass can hold. The orange band holds the published estimates.

The panels have no headline titles, because in a journal the caption carries the message.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [1.25, 1]})
fig.patch.set_facecolor("white")
for ax in (ax1, ax2):
    ax.set_facecolor("white"); ax.set_axisbelow(True)

# ---- left: Eddington ratio ----
labels = ["Surface gravity\n(Curtis et al. 2026)", "Blueshifted Hβ absorber, if at the\nescape speed (Naidu et al. 2026a)",
          f"{len(others)} other little red dots\n(Curtis et al. 2026)"]
ax1.axvspan(0.76, 1.58, ymin=0.33, ymax=1, color=AQUA, alpha=0.22, lw=0, zorder=0)
ax1.axvspan(5, 50, ymin=0.33, ymax=1, color=ZONE, lw=0, zorder=0)
ax1.text(np.sqrt(0.76*1.58), 2.62, "classic\nmodel", color=INK2, fontsize=9.5, ha="center", va="top")
ax1.text(np.sqrt(5*50), 2.62, "typical BH*\n(Sun et al. 2026)", color=INK2, fontsize=9.5, ha="center", va="top")

def point(ax, y, val, filled):
    mid, lo, hi = val
    ax.errorbar(mid, y, xerr=[[mid-lo], [hi-mid]], fmt="o", color=INK, ms=8.5, mfc=INK if filled else "white",
                mec=INK, mew=1.5, lw=2, capsize=4)
point(ax1, 2.0, RATIO_GRAV_C, False)
point(ax1, 1.12, RATIO_SPEED, True)
point(ax1, 0.88, RATIO_SPEED_C, False)
jitter = np.random.default_rng(1).uniform(-0.2, 0.2, len(others))
ax1.plot(others.phi_50, 0 + jitter, "o", color=BLUE, ms=5.5, alpha=0.65, mec="white", mew=0.6)
ax1.plot([others.phi_50.median()]*2, [-0.32, 0.32], color=INK, lw=1.5)
ax1.text(others.phi_50.median(), -0.42, "median", color=INK2, fontsize=9, ha="center", va="top")
ax1.plot([], [], "o", color=INK, ms=8.5, mfc=INK, label="my photosphere")
ax1.plot([], [], "o", color=INK, ms=8.5, mfc="white", mew=1.5, label="Curtis et al.'s photosphere")
ax1.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=2, columnspacing=1.4, handletextpad=0.3)
ax1.set_xscale("log"); ax1.set_xlim(0.08, 600); ax1.set_ylim(-0.85, 2.7)
ax1.set_yticks([2, 1, 0]); ax1.set_yticklabels(labels)
ax1.tick_params(axis="y", length=0, labelcolor=INK)
ax1.grid(axis="y", visible=False)
ax1.set_xlabel("Luminosity / Eddington luminosity")
ax1.text(-0.62, 1.04, "(a)", transform=ax1.transAxes, fontsize=12, fontweight="bold", color=INK)

# ---- right: black-hole mass in the classic quasi-star model ----
bins = np.linspace(1.5, 7.5, 121)
ax2.axvspan(1e6, 1e7, color=ORANGE, alpha=0.2, lw=0, label="published estimates")
peak = 0
for key, col, lab in [("as is", BLUE, "model as is"), ("calibrated", AQUA, "calibrated to Ball et al. (2011)")]:
    P, w = res_mine[key]
    h, edges = np.histogram(P[:, 1], bins=bins, weights=w, density=True)
    ax2.step(10**edges[:-1], h, where="post", color=col, lw=1.8, label=lab)
    peak = max(peak, h.max())
P, w = res_mine["combined"]
h, edges = np.histogram(P[:, 1], bins=bins, weights=w, density=True)
ax2.fill_between(10**edges[:-1], h, step="post", color=INK2, alpha=0.16, lw=0, label="combined")
ax2.plot([Q_MAX*M_TOTAL[1]]*2, [0, 1.25*peak], color=INK2, lw=1.2, ls="--")
ax2.text(Q_MAX*M_TOTAL[1]*0.88, 1.23*peak, "classic limit", color=INK2, fontsize=9.5, rotation=90, ha="right", va="top")
ax2.set_xscale("log"); ax2.set_xlim(1e2, 3e7); ax2.set_ylim(0, 2.05*peak)
ax2.set_xlabel("Black-hole mass (M$_\\odot$)")
ax2.set_ylabel("Posterior probability density (per dex)")
ax2.legend(loc="upper left")
ax2.text(0.0, 1.04, "(b)", transform=ax2.transAxes, fontsize=12, fontweight="bold", color=INK)

plt.tight_layout(); fig.savefig("figure1_MoM-BH1.png", dpi=200, facecolor="white"); plt.show()
n_out = int(((others.phi_50 < 0.08) | (others.phi_50 > 600)).sum())
print(f"({n_out} of the {len(others)} other little red dots fall outside the left panel's range.)")

# What stage 5 tells you

- **A near-limit envelope is allowed for MoM-BH\*-1, but not shown** (Steps 2–4). Its surface gravity and its fastest gas both give upper limits on the mass that are about as high as the Eddington mass. For the typical little red dot they are far lower. But if the gas is a fast wind, MoM-BH\*-1 is an ordinary object 9–25 times above its limit.
- **Your black-hole mass does not depend on that question** (Step 5, test 3). It goes as the fourth root of the Eddington ratio.
- **It survives the hotter published temperature** (Step 5, test 2).
- **Its upper error bar is too tight** (Step 5, test 4). Write "about 10,000 Suns; up to about 200,000 not excluded".

**Numbers for your note** (copy them from your own output above):
- Step 1: the radius.
- Step 2: the typical Eddington ratio of the 87, MoM-BH\*-1's value and range, and the percentage closer to the limit.
- Step 3: the escape speed of the near-limit envelope, the mass limit from the outflow, and the wind case.
- Step 4: the Eddington ratios for both surfaces.
- Step 5: tests 2, 3 and 4.

**How far to trust it:**
- Everything about the black hole holds only inside the classic quasi-star model. Coughlin & Begelman (2024) argue that the classic limit on the black hole is unphysical, and in the newer models the surface temperature says almost nothing about the black hole.
- The two weighing methods and their reading as upper limits are from Naidu et al. (2026b). Cite that paper for them.
- Santarelli et al. (2026) already suggested a black hole of a few million Suns for MoM-BH\*-1 from a quasi-star model. Your classic-model answer is about 100 times smaller. Say so.

**The test that could still make this a full paper:** do the side components of Hβ sit at the same speed in Hγ? Moving gas says yes; scattering generally says no. Before that, check whether Naidu et al. (2026b) or Matthee et al. (2026) already list MoM-BH\*-1 among their absorbers.